In [ ]:
# %%bash

# # unzip data files from https://github.com/IBM/mt-rag-benchmark/ to prepare

# git clone https://github.com/IBM/mt-rag-benchmark
# unzip 'mt-rag-benchmark/corpora/document_level/*.zip' -d corpora/document_level/
# unzip 'mt-rag-benchmark/corpora/passage_level/*.zip' -d corpora/passage_level/
# rm mt-rag-benchmark/corpora/*/*.zip

In [80]:
from pathlib import Path
import json
from dataclasses import dataclass
from collections import defaultdict
from typing import Self

In [84]:
@dataclass
class MTRAG_Document:
    document_id: str
    """A unique ID."""

    text: str
    """Composed of .title + linebreak + page text. May have some weird
    prefix symbols such as ï»¿\n\n.
    """

    title: str | None = None
    """Looks like a webpage title. For "cloud" corpus is ommited for all
    documents.
    """

    url: str | None = None
    """A URL, sometimes a regexp. May be omitted in some corpora."""

    domain: str | None = None
    """For "cloud" corpus may be N/A, /app-configuration or
    AnalyticsEngine.
    """

    @classmethod
    def from_json(cls, json_string: str) -> Self:
        """Some field preprocessing to unify formats across corpora."""
        data = json.loads(json_string)
        if '_id' in data:
            # field varies: sometimes _id, sometimes document_id
            assert 'id' not in data
            data['document_id'] = data.pop('_id')
        if 'metadata' in data:
            # always seems to be empty, so skip it
            assert data['metadata'] == {}
            del data['metadata']
        return cls(**data)


@dataclass
class MTRAG_Passage:
    id: str
    """A unique ID, typically formatted as {doc_id}-{start_char}-{end_char}."""

    _id: str
    """Equals .id for all passages (checked)."""

    url: str
    """A URL, sometimes a regexp (how?)."""

    title: str
    """Looks like a webpage title (?)."""

    text: str
    """Composed of .title + linebreak + page text."""

    def __post_init__(self):
        """Rules inferred from the MTRAG data."""
        assert self._id == self.id
        assert self.text.startswith(self.title + '\n')


corpora_as_documents: dict[str, list[MTRAG_Document]] = defaultdict(list)
for path in Path('mt-rag-benchmark/corpora/document_level').glob('*.jsonl'):
    for line in path.read_text().strip().split('\n'):
        corpora_as_documents[path.stem].append(
            MTRAG_Document.from_json(line)
        )


corpora_as_passages: dict[str, list[MTRAG_Passage]] = defaultdict(list)
for path in Path('mt-rag-benchmark/corpora/passage_level').glob('*.jsonl'):
    for line in path.read_text().strip().split('\n'):
        corpora_as_passages[path.stem].append(
            MTRAG_Passage(**json.loads(line))
        )

for name in corpora_as_passages:
    print(
        f'{name}:'
        f' {len(corpora_as_documents[name])} documents'
        f', {len(corpora_as_passages[name])} passages'
    )

govt: 7661 documents, 49607 passages
fiqa: 57638 documents, 61022 passages
cloud: 8578 documents, 72442 passages
clapnq: 178890 documents, 183408 passages
